# S163 — ARGOS sanitizer GO/NO-GO: LaMa (2022 baseline) vs Qwen-Image-Edit (SOTA 2026)

**Scopo**: confronto qualita' inpainting su 3 foto auto dealer reali con watermark + targa.

**Setup Colab**: Runtime -> Change runtime type -> **T4 GPU**.

**Modelli a confronto**:
| Modello | Anno | Size | Workflow | Tempo T4 free |
|---|---|---|---|---|
| LaMa (Sanster/big-lama via IOPaint) | 2022 | 200MB | erase **given mask** | ~5-15s/img |
| Qwen/Qwen-Image-Edit (Apache-2.0) | 2026 | ~20B params (CPU offload) | **prompt-based** (no mask) | ~2-5min/img |

**Workflow Luke**: upload 3 foto -> run all -> confronta output side-by-side -> decisione GO/NO-GO.

**Criteri**: GO = vincitore senza artefatti grossolani a 100% zoom su area inpainted. NO-GO entrambi -> S163-bis (BrushNet/PowerPaint v2 stesso IOPaint).

**Vincoli**: #1 verifica fattuale (Qwen-Image-Edit pipeline class + IOPaint 1.6.0 CLI verificati), #5 zero-cost (T4 free + CPU offload per Qwen), #4 critica (vedi cella finale).

## 1. Verifica GPU + memoria

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

## 2. Install dipendenze (~5-7min)

`iopaint` per LaMa baseline + `diffusers` (git ultime feature Qwen) + `accelerate` per CPU offload.

In [ ]:
!pip install -q iopaint==1.6.0
!pip install -q -U transformers accelerate
!pip install -q git+https://github.com/huggingface/diffusers

## 3. Upload 3 foto sample

Trascina dal Mac: `~/Documents/combaretrovamiauto-enterprise/sanitizer_colab/inputs/sample_A.jpg`, `sample_B.jpg`, `sample_C.jpg`.

**Alternative path immagini** (sceglile tu se preferisci foto diverse):
- `~/Documents/combaretrovamiauto-enterprise/dossiers/safe_images/argos_autoscout24_de_*.jpg` (60+ foto reali scraperate, con watermark dealer + targa)
- Drag&drop nella cella sotto. Servono esattamente 3 file, ridenominati in `sample_A.jpg`, `sample_B.jpg`, `sample_C.jpg`.

In [ ]:
from google.colab import files
import os, shutil

os.makedirs('/content/inputs', exist_ok=True)
os.makedirs('/content/masks', exist_ok=True)
os.makedirs('/content/outputs_lama', exist_ok=True)
os.makedirs('/content/outputs_qwen', exist_ok=True)

uploaded = files.upload()
for name in uploaded:
    target = name
    if not name.startswith('sample_'):
        # rinomina automaticamente se Luke non ha rinominato
        idx = ['A', 'B', 'C'][len(os.listdir('/content/inputs'))]
        target = f'sample_{idx}.jpg'
    shutil.move(name, f'/content/inputs/{target}')
    print(f'OK: /content/inputs/{target}')

assert len(os.listdir('/content/inputs')) == 3, 'servono esattamente 3 foto'
!ls -lh /content/inputs/

## 4. LaMa baseline (richiede mask)

**Genera mask manuale** disegnando bbox sui watermark/targa di OGNI foto. Apri la cella sotto, modifica le coordinate per ogni `sample_X.jpg` guardando le foto, esegui.

Formato: lista di tuple `(left, top, right, bottom)` in pixel. Coords assolute (non normalizzate). Bianco=area da inpaint.

In [ ]:
from PIL import Image, ImageDraw

# Modifica queste coords dopo aver visto le foto (apri /content/inputs/sample_X.jpg in Colab file explorer)
MASK_BBOXES = {
    'sample_A.jpg': [
        # (left, top, right, bottom) — esempi indicativi, AGGIORNA dopo aver guardato
        (500, 1000, 950, 1070),   # targa centro-basso
        (1180, 970, 1430, 1060),  # watermark angolo BR
    ],
    'sample_B.jpg': [
        (440, 880, 850, 945),
        (1040, 860, 1270, 940),
    ],
    'sample_C.jpg': [
        (440, 880, 850, 945),
        (1040, 860, 1270, 940),
    ],
}

for name, bboxes in MASK_BBOXES.items():
    img = Image.open(f'/content/inputs/{name}')
    mask = Image.new('L', img.size, 0)
    d = ImageDraw.Draw(mask)
    for bbox in bboxes:
        d.rectangle(bbox, fill=255)
    out = '/content/masks/' + name.replace('.jpg', '.png')
    mask.save(out)
    print(f'mask -> {out} (img size {img.size})')

# preview overlay mask su image per verifica visuale
from IPython.display import display
for name in MASK_BBOXES:
    img = Image.open(f'/content/inputs/{name}').convert('RGBA')
    mask = Image.open('/content/masks/' + name.replace('.jpg', '.png')).convert('L')
    overlay = Image.new('RGBA', img.size, (255, 0, 0, 0))
    overlay.paste((255, 0, 0, 120), mask=mask)
    preview = Image.alpha_composite(img, overlay)
    preview.thumbnail((600, 600))
    print(name)
    display(preview)

## 5. LaMa inpaint (CLI iopaint, ~5-15s/img su T4)

In [ ]:
!python -m iopaint run --model lama --device cuda --image /content/inputs --mask /content/masks --output /content/outputs_lama --concat
!ls -lh /content/outputs_lama/

## 6. Qwen-Image-Edit (prompt-based, no mask)

**CPU offload obbligatorio** su T4 15GB (20B params bfloat16 = ~40GB teorico).
Inference ~2-5min/img. Per 3 foto: ~10-15min totali. **Sii paziente, non killare la cella**.

In [ ]:
import torch
from diffusers import QwenImageEditPipeline
from PIL import Image
import os, time

print('Loading Qwen-Image-Edit (download ~20GB al primo run, 5-10min)...')
pipe = QwenImageEditPipeline.from_pretrained('Qwen/Qwen-Image-Edit', torch_dtype=torch.bfloat16)
pipe.enable_model_cpu_offload()  # critico per T4 free
print('Pipeline ready.')

PROMPT = 'remove all watermarks, logos, and the license plate. keep the car intact, clean background where elements were removed.'
NEGATIVE = 'distortion, blur, artifacts, ghost text'

for name in sorted(os.listdir('/content/inputs')):
    print(f'\n=== {name} ===')
    t0 = time.time()
    image = Image.open(f'/content/inputs/{name}').convert('RGB')
    out = pipe(
        image=image,
        prompt=PROMPT,
        negative_prompt=NEGATIVE,
        num_inference_steps=50,
        true_cfg_scale=4.0,
        generator=torch.manual_seed(42),
    ).images[0]
    target = f'/content/outputs_qwen/{name.replace(".jpg", ".png")}'
    out.save(target)
    print(f'  saved {target} in {int(time.time()-t0)}s')

!ls -lh /content/outputs_qwen/

## 7. Confronto side-by-side: original | LaMa | Qwen

In [ ]:
from PIL import Image
from IPython.display import display
import os

for name in sorted(os.listdir('/content/inputs')):
    orig = Image.open(f'/content/inputs/{name}').convert('RGB')
    # LaMa output con --concat ha original|mask|result orizzontale; estraggo solo result (terzo)
    lama_concat_path = f'/content/outputs_lama/{name}'
    if not os.path.exists(lama_concat_path):
        lama_concat_path = lama_concat_path.replace('.jpg', '.png')
    lama_full = Image.open(lama_concat_path).convert('RGB')
    third = lama_full.width // 3
    lama = lama_full.crop((third * 2, 0, lama_full.width, lama_full.height))
    qwen = Image.open(f'/content/outputs_qwen/{name.replace(".jpg", ".png")}').convert('RGB')
    # resize tutto a stessa altezza per comparison strip
    H = 400
    def fit(im):
        w = int(im.width * H / im.height)
        return im.resize((w, H))
    strip = Image.new('RGB', (fit(orig).width + fit(lama).width + fit(qwen).width + 20, H), 'white')
    x = 0
    for im in [fit(orig), fit(lama), fit(qwen)]:
        strip.paste(im, (x, 0))
        x += im.width + 10
    strip.save(f'/content/compare_{name.replace(".jpg", ".png")}')
    print(f'{name}: ORIGINAL | LaMa | Qwen-Image-Edit')
    display(strip)

## 8. Download zip risultati

In [ ]:
!cd /content && zip -r s163_results.zip inputs masks outputs_lama outputs_qwen compare_*.png
from google.colab import files
files.download('/content/s163_results.zip')

## 9. Decisione GO/NO-GO

Per ogni sample, valuta a 100% zoom area inpainted:
- **GO LaMa**: 3/3 risultati plausibili senza banding/halo/texture ripetuta -> S164 = aggiungi detection automatica mask
- **GO Qwen**: 3/3 risultati plausibili -> **bypass S164** (prompt-based non richiede mask), procedi a S165 wrap ARGOS production
- **NO-GO entrambi**: S163-bis = test BrushNet (`iopaint run --model brushnet`) o PowerPaint v2 (`--model powerpaint`)

## Critica strutturale piano (vincolo #4)
1. **Assunzione**: Qwen-Image-Edit prompt-based interpreta correttamente "remove watermark and license plate" su foto auto reali. Se mismatch dominio (modello trainato su scene generiche, watermark dealer DE = caso edge), fallback su mask manuale -> stesso workflow LaMa.
2. **Cosa rompe a 60gg**: Colab T4 free disconnect ~12h, Qwen download ~20GB ogni run (cache HF non persistente cross-session). Per produzione ARGOS = serve self-hosted GPU o pivot a HF Inference Endpoints paid (fuori scope MVP).
3. **Pattern errore noto**: S159-S162 stack ML pesante = rischio dependency conflicts. `pip install git+diffusers` su Colab solitamente liscio, ma `transformers` major upgrade puo' rompere altri import (pin -U accelerate prima).
4. **Sovradimensione**: 3 sample possono non bastare per discriminare LaMa vs Qwen su edge cases. Se decision è 51/49, allarga sample a 10 (foto extra in `dossiers/safe_images/`).